# Chapter 2: Pre-Model Workflow and Data Preprocessing

## 1. Chapter Summary
Kualitas data merupakan faktor penentu utama keberhasilan performa model machine learning (*garbage in, garbage out*). Bab ini berfokus pada teknik persiapan dan rekayasa data sebelum memasuki tahap pelatihan model:
- **Penanganan Missing Values**: Strategi imputasi statistik (`SimpleImputer`), tetangga terdekat (`KNNImputer`), serta regresi multivariat berulang (`IterativeImputer`).
- **Penskalaan Fitur (Feature Scaling)**: Standarisasi distribusi Gaussian (`StandardScaler`), normalisasi rentang terbatas (`MinMaxScaler`), dan normalisasi unit norm per baris (`Normalizer`).
- **Encoding Variabel Kategorikal**: Konversi data teks/kategori menjadi numerik via `OneHotEncoder`, `LabelEncoder`, serta orkestrasi kolom heterogen melalui `ColumnTransformer`.
- **Pipeline & Alur Terpadu**: Enkapsulasi tahapan transformasi data dengan `Pipeline` untuk menjaga reproduktifitas dan mencegah *data leakage*.
- **Feature Engineering & Selection**: Pembuatan fitur interaksi (`PolynomialFeatures`), diskretisasi kontinu (`KBinsDiscretizer`), serta seleksi fitur relevan menggunakan `RFE` dan `SelectFromModel`.

In [14]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Pengaturan format tampilan pandas
pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.4f}'.format)

print("Environment siap untuk eksperimen Data Preprocessing.")

Environment siap untuk eksperimen Data Preprocessing.


## 2. Theoretical Deep-Dive: Dampak Data Mentah & Imputasi Missing Values

### 2.1 Mengapa Data Perlu Diproses Awal?
Data mentah di dunia nyata umumnya memuat nilai yang hilang (*missing values*), pencilan (*outliers*), skala variabel yang timpang, serta format kategorikal yang tidak dapat langsung diproses oleh operasi aljabar linear pada model pembelajaran mesin.

### 2.2 Strategi Penanganan Nilai Hilang
1. **SimpleImputer**: Mengganti nilai kosong menggunakan estimasi statistik univariat sederhana (rata-rata/mean, median, atau modus). Cepat secara komputasi namun cenderung memangkas variansi asli fitur.
2. **KNNImputer**: Menggunakan pendekatan $k$-Nearest Neighbors dengan jarak Euclidean berbobot untuk mengisi nilai kosong berdasarkan kemiripan observasi terdekat.
3. **IterativeImputer (MICE)**: Memodelkan setiap fitur yang memiliki kekosongan nilai sebagai fungsi regresi terhadap fitur-fitur lainnya secara bergantian hingga konvergen, sehingga hubungan multivariat tetap terjaga.

In [12]:
# Pembuatan Dataset Tiruan dengan Missing Values (~20%)
np.random.seed(2024)
n_samples = 20
n_features = 10

data = {
    f"Feature{i+1}": np.random.uniform(0, 100, n_samples)
    for i in range(n_features)
}
df_missing = pd.DataFrame(data)

# Menginjeksi missing values secara acak
for col in df_missing.columns:
    mask = np.random.random(n_samples) < 0.2
    df_missing.loc[mask, col] = np.nan

print(f"Total sel hilang: {df_missing.isna().sum().sum()} dari {df_missing.size} sel ({df_missing.isna().sum().sum()/df_missing.size*100:.1f}%)")
display(df_missing.head(5))

Total sel hilang: 44 dari 200 sel (22.0%)


,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.8015,NaN,42.0098,34.6804,49.9623,NaN,NaN,89.9546,41.1524,NaN
1,69.9109,9.5542,6.4364,31.2878,37.9665,NaN,82.0056,NaN,75.7976,91.3825
2,NaN,96.0910,59.6433,84.7104,NaN,84.1090,NaN,70.2881,1.7783,4.1177
3,NaN,25.1767,83.7324,88.0231,16.8869,97.2056,84.6968,NaN,NaN,80.0780
4,20.5019,NaN,89.2486,67.6559,58.6359,78.2257,60.9116,NaN,65.1142,99.1192


In [13]:
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# 1. SimpleImputer (Mean)
mean_imputer = SimpleImputer(strategy="mean")
df_simple = pd.DataFrame(mean_imputer.fit_transform(df_missing), columns=df_missing.columns)

# 2. KNNImputer (k=2)
knn_imputer = KNNImputer(n_neighbors=2)
df_knn = pd.DataFrame(knn_imputer.fit_transform(df_missing), columns=df_missing.columns)

# 3. IterativeImputer
iter_imputer = IterativeImputer(random_state=2024, max_iter=10)
df_iter = pd.DataFrame(iter_imputer.fit_transform(df_missing), columns=df_missing.columns)

print("=== Perbandingan Hasil Pengisian Missing Values pada Baris Pertama ===")
print("SimpleImputer  :", df_simple.iloc[0, :3].values)
print("KNNImputer     :", df_knn.iloc[0, :3].values)
print("IterativeImputer:", df_iter.iloc[0, :3].values)

=== Perbandingan Hasil Pengisian Missing Values pada Baris Pertama ===
SimpleImputer  : [58.80145189 53.86466147 42.00981389]
KNNImputer     : [58.80145189 48.95404283 42.00981389]
IterativeImputer: [58.80145189 54.36500811 42.00981389]


## 3. Penskalaan Fitur (Feature Scaling)

Fitur dengan rentang skala yang berbeda drastis dapat mendistorsi fungsi jarak (seperti pada KNN/SVM) atau memperlambat konvergensi *gradient descent*.
- **StandardScaler**: Mentransformasikan data agar memiliki nilai $\mu = 0$ dan $\sigma = 1$:
  $$z = \frac{x - \mu}{\sigma}$$
- **MinMaxScaler**: Mengompresi nilai ke dalam interval tertutup $[0, 1]$:
  $$x' = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$
- **Normalizer**: Melakukan penskalaan per baris sampel (bukan kolom) sehingga panjang vektor L2-norm sama dengan 1.

In [11]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, Normalizer

# Menggunakan data hasil IterativeImputer
scaler_std = StandardScaler()
df_scaled_std = pd.DataFrame(scaler_std.fit_transform(df_iter), columns=df_iter.columns)

scaler_minmax = MinMaxScaler()
df_scaled_minmax = pd.DataFrame(scaler_minmax.fit_transform(df_iter), columns=df_iter.columns)

normalizer = Normalizer(norm='l2')
df_normalized = pd.DataFrame(normalizer.fit_transform(df_iter), columns=df_iter.columns)

print("=== Statistik Setelah StandardScaler (Rata-rata ~ 0, Std ~ 1) ===")
print("Mean:", np.round(df_scaled_std.mean().values[:3], 4))
print("Std :", np.round(df_scaled_std.std(ddof=0).values[:3], 4))

print("\n=== Rentang Nilai Setelah MinMaxScaler (Min = 0, Max = 1) ===")
print("Min:", df_scaled_minmax.min().values[:3])
print("Max:", df_scaled_minmax.max().values[:3])

print("\n=== Verifikasi Unit Norm pada Normalizer (Baris) ===")
print("L2 Norm per baris (3 sampel teratas):", np.round(np.sqrt((df_normalized**2).sum(axis=1))[:3].values, 4))

=== Statistik Setelah StandardScaler (Rata-rata ~ 0, Std ~ 1) ===
Mean: [-0.  0.  0.]
Std : [1. 1. 1.]

=== Rentang Nilai Setelah MinMaxScaler (Min = 0, Max = 1) ===
Min: [0. 0. 0.]
Max: [1. 1. 1.]

=== Verifikasi Unit Norm pada Normalizer (Baris) ===
L2 Norm per baris (3 sampel teratas): [1. 1. 1.]


## 4. Encoding Variabel Kategorikal & ColumnTransformer

- **OneHotEncoder**: Memecah variabel kategori menjadi beberapa kolom biner. Pilihan terbaik untuk kategori nominal agar tidak memunculkan hierarki nilai semu.
- **LabelEncoder**: Mengubah kategori teks menjadi integer urut ($0, 1, 2, \dots$).
- **ColumnTransformer**: Modul penting untuk menerapkan teknik transformasi berbeda pada jenis kolom berbeda secara simultan (misal penskalaan pada numerik dan one-hot pada kategorikal).

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Membuat dataset tiruan campuran (numerik + kategorikal)
np.random.seed(2024)
mixed_df = pd.DataFrame({
    "Age": np.random.randint(22, 60, size=20),
    "Salary": np.round(np.random.normal(55000, 12000, size=20), 2),
    "Experience": np.random.randint(1, 15, size=20),
    "Department": np.random.choice(["IT", "HR", "Sales", "Finance"], size=20),
    "Position": np.random.choice(["Junior", "Senior", "Manager"], size=20)
})

# Inisialisasi ColumnTransformer
ct = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), ["Age", "Salary", "Experience"]),
        ("cat", OneHotEncoder(), ["Department", "Position"])
    ],
    remainder="passthrough"
)

transformed_array = ct.fit_transform(mixed_df)
cat_cols = list(ct.named_transformers_["cat"].get_feature_names_out(["Department", "Position"]))
feature_names = ["Age_scaled", "Salary_scaled", "Experience_scaled"] + cat_cols

df_prepared = pd.DataFrame(transformed_array, columns=feature_names)
print(f"Bentuk data asal: {mixed_df.shape} -> Bentuk setelah diproses: {df_prepared.shape}")
display(df_prepared.head(5))

Bentuk data asal: (20, 5) -> Bentuk setelah diproses: (20, 10)


,Age_scaled,Salary_scaled,Experience_scaled,Department_Finance,Department_HR,Department_IT,Department_Sales,Position_Junior,Position_Manager,Position_Senior
0,-1.0392,-1.4416,-0.2484,1.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000
1,0.8574,-1.4370,-0.0324,0.0000,0.0000,0.0000,1.0000,1.0000,0.0000,0.0000
2,-1.6714,-0.4344,0.1836,1.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000
3,0.4623,0.9702,1.2638,0.0000,0.0000,0.0000,1.0000,1.0000,0.0000,0.0000
4,1.1735,-0.7619,1.2638,1.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000


## 5. Implementasi Pipeline & Pencegahan Data Leakage

`Pipeline` mengintegrasikan alur pembersihan data dan pemodelan dalam satu kesatuan.
Aturan krusial: dataset harus dipisahkan terlebih dahulu menggunakan `train_test_split()` sebelum transformasi diterapkan. `fit()` dan `fit_transform()` hanya dijalankan pada data latih, sedangkan data uji dievaluasi semata-mata dengan `.transform()` untuk menghindari kebocoran parameter statistik data uji ke data latih (*data leakage*).

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Memisahkan matriks fitur dan target hipotetis
X = df_prepared.iloc[:, :-1]
y = np.where(df_prepared.iloc[:, -1] > 0.5, 1, 0)

# Pembagian data latih dan uji
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=2024)

# Membangun Pipeline terpadu
ml_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(random_state=2024))
])

# Melatih pipeline
ml_pipeline.fit(X_train, y_train)

train_accuracy = ml_pipeline.score(X_train, y_train)
test_accuracy = ml_pipeline.score(X_test, y_test)

print("Tahapan Pipeline:", [step[0] for step in ml_pipeline.steps])
print(f"Akurasi Data Latih : {train_accuracy:.4f}")
print(f"Akurasi Data Uji   : {test_accuracy:.4f}")

Tahapan Pipeline: ['imputer', 'scaler', 'classifier']
Akurasi Data Latih : 1.0000
Akurasi Data Uji   : 1.0000


## 6. Feature Engineering & Feature Selection

1. **PolynomialFeatures**: Membentuk kombinasi polinomial dan interaksi antar-fitur untuk menangkap dinamika non-linear data.
2. **KBinsDiscretizer**: Mengonversi nilai numerik kontinu ke dalam interval atau kategori diskret berbasis binning.
3. **Recursive Feature Elimination (RFE)** & **SelectFromModel**: Mengeliminasi fitur yang tidak informatif berdasarkan pembobotan atau koefisien kepentingan dari estimator acuan.

In [8]:
from sklearn.preprocessing import PolynomialFeatures, KBinsDiscretizer
from sklearn.feature_selection import RFE, SelectFromModel
from sklearn.linear_model import LinearRegression

# 1. Polynomial Features pada data latih (derajat 2)
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_train)
print(f"Fitur Awal: {X_train.shape[1]} -> Fitur Polinomial: {X_poly.shape[1]}")

# 2. Diskretisasi dengan KBinsDiscretizer
discretizer = KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='uniform')
X_binned = discretizer.fit_transform(X_train.iloc[:, :3])
print(f"Sampel hasil binning (3 fitur awal):\n{X_binned[:3]}")

# 3. Seleksi Fitur dengan SelectFromModel
estimator = LinearRegression()
selector = SelectFromModel(estimator=estimator, threshold="mean")
selector.fit(X_train, y_train)

selected_features = X_train.columns[selector.get_support()].tolist()
print("\n=== Hasil Seleksi Fitur (SelectFromModel) ===")
print("Fitur Terpilih:", selected_features)

Fitur Awal: 9 -> Fitur Polinomial: 54
Sampel hasil binning (3 fitur awal):
[[0. 2. 1.]
 [2. 1. 2.]
 [0. 0. 1.]]

=== Hasil Seleksi Fitur (SelectFromModel) ===
Fitur Terpilih: ['Position_Junior', 'Position_Manager']


## 7. Analysis & Output Conclusions
1. **Preservasi Nilai Melalui Imputasi**: Imputasi berbasis model seperti `IterativeImputer` dan `KNNImputer` terbukti mempertahankan korelasi antar-fitur lebih baik dibanding pengisian nilai konstan/rata-rata (`SimpleImputer`).
2. **Kesesuaian Penskalaan**: Penggunaan `StandardScaler` cocok untuk algoritma berbasis gradien dan jarak yang mengasumsikan data terdistribusi normal, sedangkan `MinMaxScaler` unggul pada model yang memerlukan batas angka mutlak $[0, 1]$.
3. **Pemberdayaan Pipeline**: Pipeline terpadu menjamin seluruh transformasi otomatis diterapkan ke data baru tanpa intervensi manual, mengeliminasi kesalahan komputasi dan risiko *data leakage*.
4. **Efisiensi Seleksi Fitur**: Seleksi fitur menyaring variabel berkorelasi rendah, sehingga mereduksi dimensi data, mempercepat proses training, serta mengurangi risiko *overfitting*.